# 9장 — 여럿이 하기: 멀티에이전트 협업

웹 챕터(9장)와 **같은 예제, 같은 수치**를 파이썬으로 직접 계산합니다 — 전부 해석적(난수 없음)이라
웹 실험실(TeamLab)과 결과가 정확히 일치해요.

> 🤖 **AI 튜터와 함께 공부하세요** — 아래 셀을 실행하면 화면 오른쪽 아래에 튜터 버튼이 뜹니다.
> 로컬 Ollama 가 꺼져 있어도 노트북 학습에는 지장이 없습니다.

In [ ]:
from tutor import chat
chat('ch9')  # 이 챕터를 아는 AI 튜터 (로컬 Ollama — 없어도 학습엔 지장 없음)

## 1. 네 가지 팀 구성 — 같은 일, 다른 구조

걸음 정확도 90%·10단계 일(워밍업의 그 보고서)을 네 구성으로 맡깁니다.

| 구성 | 정의 | 공식 |
|---|---|---|
| A 혼자 | 한 명이 10단계 전부 | $0.9^{10}$ |
| B 분업(q=0.95) | 2명이 5단계씩 + 전달 2회 | $0.9^{10}\times q^2$ |
| C 분업(q=0.99) | 전달에 정성을 들인 분업 | $0.9^{10}\times 0.99^2$ |
| D 작성+검토 | 검토 통과만 제출, 거부 시 재작성 1회 | 아래 계산 |

D의 정책: 작성(성공확률 $p=0.9^{10}$) → 검토(민감도 $s=0.9$, 특이도 $t=0.8$) →
통과하면 제출, 거부되면 **재작성 1회** 후 재검토(최대 2회 작성, 통과분만 제출).
최종 성공 = 제출됐고 정답: $p\,s\,(1+r)$, 거부율 $r = p(1-s)+(1-p)\,t$.

In [ ]:
p = 0.9 ** 10                      # A. 혼자 — 워밍업의 34.9%
B = p * 0.95 ** 2                  # B. 분업 + 전달 2회(q=0.95)
C = p * 0.99 ** 2                  # C. 분업 + 전달 2회(q=0.99)
s, t = 0.9, 0.8                    # D. 검토자 — 민감도·특이도
r = p * (1 - s) + (1 - p) * t      #    1회차 거부율
D = p * s * (1 + r)                #    최대 2회 작성, 통과분만 제출

for name, v in [('A 혼자', p), ('B 분업 q=0.95', B), ('C 분업 q=0.99', C), ('D 작성+검토', D)]:
    print(f'{name:14s} {v:.4f}  ({v*100:.1f}%)')

assert abs(p - 0.3486784401) < 1e-9
assert abs(B - 0.3146822922) < 1e-9   # 31.5% — 혼자보다 낮다!
assert abs(C - 0.3417397391) < 1e-9   # 34.2% — 전달 99%여도 못 넘는다 (q²<1)
assert abs(r - 0.5559250919) < 1e-9
assert abs(D - 0.4882657806) < 1e-9   # 48.8% — 구조가 만든 역전
assert B < p and C < p and D > p      # 나누기만 하면 지고, 걸러내면 이긴다
print('✅ 구성 4종 자가검증 통과 — 머릿수가 아니라 구조')

## 2. 합의의 상관, 역할 분리의 베이즈

**합의**는 5장 다수결의 팀 버전 — 같은 모델 3대(상관 ρ 높음)와 다양한 팀(ρ 낮음)의 차이를
5장 상관 모형($P = \rho\,p + (1-\rho)\,\text{이항합}$)으로 계산합니다.

**역할 분리** — 검토를 통과한 제출물의 신뢰도는 6장 베이즈:
$P(\text{정답}\mid\text{통과}) = \dfrac{p\,s}{p\,s + (1-p)(1-t)}$

In [ ]:
import math

# 합의 — 한 표 70%, 3표 다수결 (5장 재사용)
p1 = 0.7
maj3 = 3 * p1**2 * (1 - p1) + p1**3
same_team = 0.7 * p1 + 0.3 * maj3   # 같은 모델 3대 (ρ=0.7)
div_team = 0.1 * p1 + 0.9 * maj3    # 다양한 팀 (ρ=0.1)
print(f'독립 3표 다수결 {maj3:.4f} | 같은 모델 3대(ρ=0.7) {same_team:.4f} | 다양한 팀(ρ=0.1) {div_team:.4f}')
assert abs(maj3 - 0.784) < 1e-9
assert abs(same_team - 0.7252) < 1e-9   # 72.5% — 닮은 팀은 유효 표가 준다
assert abs(div_team - 0.7756) < 1e-9    # 77.6%

# 역할 분리 — 제출물 신뢰도(베이즈, 6장)
p = 0.9 ** 10
s, t = 0.9, 0.8
precision = p * s / (p * s + (1 - p) * (1 - t))
print(f'검토 통과 제출물의 신뢰도: {precision:.4f} ({precision*100:.1f}%)')
assert abs(precision - 0.7066613997) < 1e-9   # 34.9% → 70.7%

# 분업이 확률로 이기는 유일한 경로 — 걸음 자체가 좋아질 때 (컨텍스트 분리)
boosted = 0.92 ** 10 * 0.99 ** 2
print(f'걸음 92% 분업(q=0.99): {boosted:.4f} ({boosted*100:.1f}%) — 혼자({0.9**10*100:.1f}%)를 역전')
assert abs(boosted - 0.4257441240) < 1e-9
assert boosted > 0.9 ** 10
print('✅ 합의·베이즈·역전 시나리오 자가검증 통과')

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

qs = [0.90 + 0.005 * i for i in range(21)]
solo = 0.9 ** 10 * 100
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(qs, [0.9 ** 10 * q * q * 100 for q in qs], marker='o', ms=3, color='#f43f5e',
        label='분업 (걸음 90%) — 항상 혼자보다 아래')
ax.plot(qs, [0.92 ** 10 * q * q * 100 for q in qs], marker='o', ms=3, color='#10b981',
        label='분업 (걸음 92% — 컨텍스트 분리 이득)')
ax.axhline(solo, ls='--', lw=1.2, color='#6366f1', label=f'혼자 10단계 ({solo:.1f}%)')
ax.set_xlabel('전달 정확도 q')
ax.set_ylabel('과업 성공률 (%)')
ax.set_title('분업의 손익분기 — q(전달)로는 못 이기고, 걸음(p)이 올라야 이긴다')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 3. 오늘의 발견

- **전달도 걸음이다** — 분업은 $p^{10}\times q^2$: q=0.99여도 혼자(34.9%)를 못 넘는다(34.2%).
- **구조가 역전을 만든다** — 같은 2명이라도 작성+검토(거부 시 재작성 1회)는 48.8%, 제출물 신뢰도는 70.7%(베이즈).
- **닮은 팀은 유효 머릿수가 준다** — 같은 모델 3대의 합의 72.5% vs 다양한 팀 77.6% (5장 상관의 재등장).
- 분업이 확률로 이기는 유일한 경로는 **걸음 자체의 개선**(작은 컨텍스트·전문화): 걸음 92%면 42.6%로 역전.

다음 장(10장 — 기억의 경제학)에서 그 "작은 컨텍스트"의 값어치를 계산합니다.
**웹 챕터로 돌아가 연습 10문제를 풀면 진도가 채워져요.**